## IMPORT dan Konfigurasi 

In [1]:
import requests
import pandas as pd
import json
import os
import time

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

RAW_DIR = "dataset/dataset_raw"

TARGETS = {
    "EGFR": "CHEMBL203",
    "BRAF": "CHEMBL5145"
}

PAGE_SIZE = 1000

os.makedirs(RAW_DIR, exist_ok=True)

print("Raw data directory:", RAW_DIR)

Raw data directory: dataset/dataset_raw


In [5]:
# ============================================================
# FETCH ALL ACTIVITY RECORDS FROM CHEMBL
# ============================================================

def fetch_chembl_activity(target_chembl_id, page_size=1000):
    """
    Fetch all activity records for a specific ChEMBL target.

    IMPORTANT:
    No scientific filtering is applied here.
    """

    all_activities = []
    offset = 0

    while True:

        params = {
        "target_chembl_id": target_chembl_id,
        "standard_type": "IC50",
        "limit": page_size,
        "offset": offset,
        "order_by": "activity_id"
    }

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        activities = data.get("activities", [])

        if not activities:
            break

        all_activities.extend(activities)

        print(
            f"Target {target_chembl_id} | "
            f"Fetched: {len(all_activities):,}"
        )

        page_meta = data.get("page_meta", {})

        if not page_meta.get("next"):
            break

        offset += page_size

        time.sleep(0.2)

    return all_activities

In [6]:
# ============================================================
# FETCH EGFR
# ============================================================

egfr_raw = fetch_chembl_activity(
    target_chembl_id=TARGETS["EGFR"],
    page_size=PAGE_SIZE
)

print("\nEGFR fetching completed.")
print(f"Total records: {len(egfr_raw):,}")

Target CHEMBL203 | Fetched: 1,000
Target CHEMBL203 | Fetched: 2,000
Target CHEMBL203 | Fetched: 3,000
Target CHEMBL203 | Fetched: 4,000
Target CHEMBL203 | Fetched: 5,000
Target CHEMBL203 | Fetched: 6,000
Target CHEMBL203 | Fetched: 7,000
Target CHEMBL203 | Fetched: 8,000
Target CHEMBL203 | Fetched: 9,000
Target CHEMBL203 | Fetched: 10,000
Target CHEMBL203 | Fetched: 11,000
Target CHEMBL203 | Fetched: 12,000
Target CHEMBL203 | Fetched: 13,000
Target CHEMBL203 | Fetched: 14,000
Target CHEMBL203 | Fetched: 15,000
Target CHEMBL203 | Fetched: 16,000
Target CHEMBL203 | Fetched: 17,000
Target CHEMBL203 | Fetched: 18,000
Target CHEMBL203 | Fetched: 19,000
Target CHEMBL203 | Fetched: 20,000
Target CHEMBL203 | Fetched: 21,000
Target CHEMBL203 | Fetched: 22,000
Target CHEMBL203 | Fetched: 23,000
Target CHEMBL203 | Fetched: 24,000
Target CHEMBL203 | Fetched: 25,000
Target CHEMBL203 | Fetched: 26,000
Target CHEMBL203 | Fetched: 26,600

EGFR fetching completed.
Total records: 26,600


In [7]:
# ============================================================
# FETCH BRAF
# ============================================================

braf_raw = fetch_chembl_activity(
    target_chembl_id=TARGETS["BRAF"],
    page_size=PAGE_SIZE
)

print("\nBRAF fetching completed.")
print(f"Total records: {len(braf_raw):,}")

Target CHEMBL5145 | Fetched: 1,000
Target CHEMBL5145 | Fetched: 2,000
Target CHEMBL5145 | Fetched: 3,000
Target CHEMBL5145 | Fetched: 4,000
Target CHEMBL5145 | Fetched: 5,000
Target CHEMBL5145 | Fetched: 6,000
Target CHEMBL5145 | Fetched: 7,000
Target CHEMBL5145 | Fetched: 8,000
Target CHEMBL5145 | Fetched: 9,000
Target CHEMBL5145 | Fetched: 10,000
Target CHEMBL5145 | Fetched: 11,000
Target CHEMBL5145 | Fetched: 11,017

BRAF fetching completed.
Total records: 11,017


In [8]:
# ============================================================
# SAVE RAW JSON
# ============================================================

egfr_json_path = os.path.join(
    RAW_DIR,
    "EGFR_ChEMBL_raw.json"
)

braf_json_path = os.path.join(
    RAW_DIR,
    "BRAF_ChEMBL_raw.json"
)

with open(egfr_json_path, "w", encoding="utf-8") as f:
    json.dump(
        egfr_raw,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(braf_json_path, "w", encoding="utf-8") as f:
    json.dump(
        braf_raw,
        f,
        ensure_ascii=False,
        indent=2
    )

print("JSON files saved.")

JSON files saved.


In [9]:
# ============================================================
# CONVERT RAW DATA TO DATAFRAME
# ============================================================

df_egfr_raw = pd.DataFrame(egfr_raw)
df_braf_raw = pd.DataFrame(braf_raw)

print("EGFR:", df_egfr_raw.shape)
print("BRAF:", df_braf_raw.shape)

EGFR: (26600, 47)
BRAF: (11017, 47)


In [10]:
# ============================================================
# SAVE RAW CSV
# ============================================================

egfr_csv_path = os.path.join(
    RAW_DIR,
    "EGFR_ChEMBL_raw.csv"
)

braf_csv_path = os.path.join(
    RAW_DIR,
    "BRAF_ChEMBL_raw.csv"
)

df_egfr_raw.to_csv(
    egfr_csv_path,
    index=False
)

df_braf_raw.to_csv(
    braf_csv_path,
    index=False
)

print("CSV files saved.")

CSV files saved.
